# 01 — Ingest and Embed

This notebook ingests regulatory PDF documents from MinIO, chunks and embeds them using `all-MiniLM-L6-v2`, and loads the resulting vectors into Milvus.

**Run this notebook once per environment, or whenever the document corpus changes.**

## Prerequisites
- MinIO tenant running and seeded with PDFs in the `documents` bucket
- Milvus standalone running in the `sovereign-rag` namespace
- All environment variables set in the workbench (see below)

## Environment variables required
Set these in the RHOAI workbench environment before running:

| Variable | Description |
|---|---|
| `MINIO_ENDPOINT` | MinIO service URL — e.g. `http://minio.sovereign-rag.svc.cluster.local:9000` |
| `MINIO_ACCESS_KEY` | MinIO access key |
| `MINIO_SECRET_KEY` | MinIO secret key |
| `MILVUS_HOST` | Milvus service host — e.g. `milvus.sovereign-rag.svc.cluster.local` |
| `MILVUS_PORT` | Milvus gRPC port — default `19530` |

## Cell 1 — Install dependencies

Run this cell first. The RHOAI standard data science image includes PyTorch and common ML libraries but not MinIO, Milvus, or PDF parsing libraries.

In [ ]:
# Install required libraries
# PyMuPDF: PDF parsing — reliable on dense regulatory documents
# pymilvus: Milvus client
# minio: MinIO Python client
# sentence-transformers: embedding model
# langchain-text-splitters: chunking utilities

import sys

!{sys.executable} -m pip install --quiet \
    PyMuPDF==1.24.0 \
    pymilvus==2.4.3 \
    minio==7.2.7 \
    sentence-transformers==3.0.1 \
    langchain-text-splitters==0.2.4 \
    tqdm==4.66.4 \

print("Dependencies installed.")

## Cell 2 — Configuration

All configuration is sourced from environment variables. No credentials in the notebook.

In [ ]:
import os

# MinIO
MINIO_ENDPOINT    = os.environ.get("MINIO_ENDPOINT", "minio.sovereign-rag.svc.cluster.local:9000")
MINIO_ACCESS_KEY  = os.environ["MINIO_ACCESS_KEY"]   # fail fast if not set
MINIO_SECRET_KEY  = os.environ["MINIO_SECRET_KEY"]   # fail fast if not set
MINIO_BUCKET      = os.environ.get("MINIO_BUCKET", "documents")
MINIO_USE_SSL     = os.environ.get("MINIO_USE_SSL", "false").lower() == "true"

# Milvus
MILVUS_HOST       = os.environ.get("MILVUS_HOST", "milvus.sovereign-rag.svc.cluster.local")
MILVUS_PORT       = int(os.environ.get("MILVUS_PORT", "19530"))
MILVUS_COLLECTION = "regulatory_docs"

# Chunking
CHUNK_SIZE        = 512    # tokens
CHUNK_OVERLAP     = 50     # tokens

# Embedding
EMBEDDING_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM     = 384    # all-MiniLM-L6-v2 output dimension

# Local temp directory for downloaded PDFs
PDF_TEMP_DIR      = "/tmp/regulatory-docs"

print("Configuration loaded.")
print(f"  MinIO endpoint : {MINIO_ENDPOINT}")
print(f"  MinIO bucket   : {MINIO_BUCKET}")
print(f"  Milvus host    : {MILVUS_HOST}:{MILVUS_PORT}")
print(f"  Collection     : {MILVUS_COLLECTION}")
print(f"  Chunk size     : {CHUNK_SIZE} tokens, {CHUNK_OVERLAP} overlap")
print(f"  Embedding model: {EMBEDDING_MODEL} (dim={EMBEDDING_DIM})")

## Cell 3 — Download PDFs from MinIO

Pulls all PDFs from the `documents` bucket into a local temp directory for processing.

In [ ]:
import os
from minio import Minio
from minio.error import S3Error
from pathlib import Path

# Create temp directory
Path(PDF_TEMP_DIR).mkdir(parents=True, exist_ok=True)

# Initialise MinIO client
minio_client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=MINIO_USE_SSL
)

# Verify bucket exists
if not minio_client.bucket_exists(MINIO_BUCKET):
    raise RuntimeError(f"Bucket '{MINIO_BUCKET}' does not exist in MinIO. "
                       f"Check the Ansible seeding step completed successfully.")

# List and download all PDFs
objects = list(minio_client.list_objects(MINIO_BUCKET, recursive=True))
pdf_objects = [o for o in objects if o.object_name.endswith(".pdf")]

if not pdf_objects:
    raise RuntimeError(f"No PDFs found in bucket '{MINIO_BUCKET}'. "
                       f"Check the Ansible seeding step completed successfully.")

print(f"Found {len(pdf_objects)} PDF(s) in bucket '{MINIO_BUCKET}':")

downloaded_files = []
for obj in pdf_objects:
    local_path = os.path.join(PDF_TEMP_DIR, os.path.basename(obj.object_name))
    minio_client.fget_object(MINIO_BUCKET, obj.object_name, local_path)
    downloaded_files.append(local_path)
    print(f"  Downloaded: {obj.object_name} -> {local_path}")

print(f"\nAll PDFs downloaded to {PDF_TEMP_DIR}")

## Cell 4 — Parse and chunk PDFs

Uses PyMuPDF for text extraction and LangChain's `RecursiveCharacterTextSplitter` for chunking.

Each chunk retains source metadata (filename, page number) for attribution in query results.

In [ ]:
import fitz  # PyMuPDF
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm

def extract_text_from_pdf(pdf_path: str) -> list[dict]:
    """
    Extracts text from each page of a PDF.
    Returns a list of dicts with 'text', 'source', and 'page' keys.
    """
    pages = []
    doc = fitz.open(pdf_path)
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text").strip()
        if text:  # skip blank pages
            pages.append({
                "text": text,
                "source": os.path.basename(pdf_path),
                "page": page_num
            })
    doc.close()
    return pages


# Initialise splitter
# RecursiveCharacterTextSplitter respects paragraph and sentence boundaries
# before falling back to character splits — better semantic coherence than
# a naive fixed-character split.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

all_chunks = []  # list of dicts: {text, source, page, chunk_id}

for pdf_path in tqdm(downloaded_files, desc="Parsing PDFs"):
    pages = extract_text_from_pdf(pdf_path)
    for page_data in pages:
        chunks = splitter.split_text(page_data["text"])
        for chunk_text in chunks:
            all_chunks.append({
                "text": chunk_text,
                "source": page_data["source"],
                "page": page_data["page"]
            })

print(f"\nChunking complete.")
print(f"  Total chunks : {len(all_chunks)}")
print(f"  From files   : {len(downloaded_files)}")
print(f"\nSample chunk:")
print(f"  Source : {all_chunks[0]['source']}, page {all_chunks[0]['page']}")
print(f"  Text   : {all_chunks[0]['text'][:200]}...")

## Cell 5 — Load embedding model

Loads `all-MiniLM-L6-v2` from HuggingFace. This runs on CPU in the workbench — no GPU needed for embedding generation.

In [ ]:
from sentence_transformers import SentenceTransformer

print(f"Loading embedding model: {EMBEDDING_MODEL}")
print("This may take a minute on first run — model downloads from HuggingFace.")
print("Subsequent runs use the cached model.")

embedding_model = SentenceTransformer(EMBEDDING_MODEL)

# Verify output dimension matches our configuration
test_embedding = embedding_model.encode(["test"])
actual_dim = test_embedding.shape[1]
assert actual_dim == EMBEDDING_DIM, \
    f"Embedding dimension mismatch: expected {EMBEDDING_DIM}, got {actual_dim}"

print(f"\nEmbedding model loaded.")
print(f"  Output dimension: {actual_dim}")

## Cell 6 — Generate embeddings

Encodes all chunks in batches. Batching avoids memory pressure on larger corpora.

In [ ]:
import numpy as np

BATCH_SIZE = 64  # adjust down if workbench runs out of memory

print(f"Generating embeddings for {len(all_chunks)} chunks (batch size: {BATCH_SIZE})...")
print("This runs on CPU — expect approximately 1-2 minutes per 1000 chunks.")

texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedding_model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True  # normalise for cosine similarity in Milvus
)

print(f"\nEmbedding generation complete.")
print(f"  Shape : {embeddings.shape}")
print(f"  dtype : {embeddings.dtype}")

## Cell 7 — Connect to Milvus and create collection

Creates the `regulatory_docs` collection if it does not already exist.
If it exists and `FORCE_RECREATE` is set to `true`, drops and recreates it — useful for reingesting an updated corpus.

In [ ]:
from pymilvus import (
    connections,
    utility,
    Collection,
    CollectionSchema,
    FieldSchema,
    DataType
)

# Allow forced recreation — useful when reingesting an updated corpus
FORCE_RECREATE = os.environ.get("FORCE_RECREATE", "false").lower() == "true"

# Connect to Milvus
connections.connect(
    alias="default",
    host=MILVUS_HOST,
    port=MILVUS_PORT
)
print(f"Connected to Milvus at {MILVUS_HOST}:{MILVUS_PORT}")

# Drop collection if force recreate requested
if FORCE_RECREATE and utility.has_collection(MILVUS_COLLECTION):
    utility.drop_collection(MILVUS_COLLECTION)
    print(f"Dropped existing collection '{MILVUS_COLLECTION}' (FORCE_RECREATE=true)")

# Create collection if it does not exist
if not utility.has_collection(MILVUS_COLLECTION):
    schema = CollectionSchema(
        fields=[
            FieldSchema(
                name="id",
                dtype=DataType.INT64,
                is_primary=True,
                auto_id=True
            ),
            FieldSchema(
                name="embedding",
                dtype=DataType.FLOAT_VECTOR,
                dim=EMBEDDING_DIM
            ),
            FieldSchema(
                name="text",
                dtype=DataType.VARCHAR,
                max_length=4096
            ),
            FieldSchema(
                name="source",
                dtype=DataType.VARCHAR,
                max_length=512
            ),
            FieldSchema(
                name="page",
                dtype=DataType.INT64
            )
        ],
        description="Regulatory document chunks for sovereign RAG"
    )

    collection = Collection(
        name=MILVUS_COLLECTION,
        schema=schema
    )
    print(f"Created collection '{MILVUS_COLLECTION}'")
else:
    collection = Collection(name=MILVUS_COLLECTION)
    print(f"Using existing collection '{MILVUS_COLLECTION}'")

print(f"  Schema fields: {[f.name for f in collection.schema.fields]}")

## Cell 8 — Insert vectors into Milvus

Inserts chunks and embeddings in batches. Milvus recommends batch sizes of 1000–5000 for optimal throughput.

In [ ]:
INSERT_BATCH_SIZE = 1000

total_inserted = 0
num_batches = (len(all_chunks) + INSERT_BATCH_SIZE - 1) // INSERT_BATCH_SIZE

print(f"Inserting {len(all_chunks)} vectors in {num_batches} batch(es)...")

for i in range(0, len(all_chunks), INSERT_BATCH_SIZE):
    batch_chunks = all_chunks[i:i + INSERT_BATCH_SIZE]
    batch_embeddings = embeddings[i:i + INSERT_BATCH_SIZE]

    data = [
        batch_embeddings.tolist(),                        # embedding
        [c["text"] for c in batch_chunks],               # text
        [c["source"] for c in batch_chunks],             # source
        [c["page"] for c in batch_chunks],               # page
    ]

    insert_result = collection.insert(data)
    total_inserted += len(insert_result.primary_keys)
    print(f"  Batch {i // INSERT_BATCH_SIZE + 1}/{num_batches}: "
          f"inserted {len(insert_result.primary_keys)} vectors")

collection.flush()  # ensure all data is persisted before indexing
print(f"\nInsert complete. Total vectors in collection: {total_inserted}")

## Cell 9 — Build vector index

Creates an IVF_FLAT index on the embedding field.

`IVF_FLAT` is a good default for demo-scale collections (tens of thousands of vectors). It provides exact nearest-neighbour results within each partition and is straightforward to reason about. For larger production collections, consider `HNSW` for better query latency at scale.

In [ ]:
index_params = {
    "metric_type": "COSINE",   # consistent with normalize_embeddings=True above
    "index_type": "IVF_FLAT",
    "params": {
        "nlist": 128           # number of cluster units — reasonable for demo scale
    }
}

print("Building vector index...")
collection.create_index(
    field_name="embedding",
    index_params=index_params
)

collection.load()  # load collection into memory for querying

print("Index built and collection loaded into memory.")
print(f"  Index type  : IVF_FLAT")
print(f"  Metric      : COSINE")
print(f"  nlist       : 128")

## Cell 10 — Verification

Runs a test query to confirm the pipeline is working end-to-end before moving to notebook 02.

In [ ]:
TEST_QUERY = "What are the capital adequacy requirements for banks?"

print(f"Running verification query: '{TEST_QUERY}'")
print("-" * 60)

# Embed the test query
query_embedding = embedding_model.encode(
    [TEST_QUERY],
    normalize_embeddings=True
).tolist()

# Search Milvus
search_params = {
    "metric_type": "COSINE",
    "params": {"nprobe": 16}  # number of cluster units to search
}

results = collection.search(
    data=query_embedding,
    anns_field="embedding",
    param=search_params,
    limit=3,
    output_fields=["text", "source", "page"]
)

print(f"Top 3 results:\n")
for rank, hit in enumerate(results[0], start=1):
    print(f"  [{rank}] Score  : {hit.score:.4f}")
    print(f"      Source : {hit.entity.get('source')}, page {hit.entity.get('page')}")
    print(f"      Text   : {hit.entity.get('text')[:200]}...")
    print()

print("-" * 60)
print("Verification complete. Proceed to notebook 02-rag-query.ipynb.")

## Cell 11 — Cleanup

Removes locally downloaded PDFs from the workbench temp directory. Vectors remain in Milvus.

In [ ]:
import shutil

shutil.rmtree(PDF_TEMP_DIR, ignore_errors=True)
print(f"Removed temporary PDF directory: {PDF_TEMP_DIR}")
print("Milvus collection and vectors are persisted — no action needed.")
print("\nNext step: open 02-rag-query.ipynb")